# Lab 3\. Training the DigitClassifier: From Gradients to a Trained Model

## Overview

In Chapter 2 you built the `DigitClassifier` and learned to read its structure and parameters. Chapter 3 explained how a model actually improves: autograd computes a gradient for every parameter, a loss function turns "how wrong the model is" into a single number, and an optimizer uses the gradients to update the parameters. In this lab you put those pieces together and watch the model learn.

You will work through the training chain one piece at a time and then assemble it. First you will run a small computation through autograd and read the gradients it produces. Then you will compute two loss functions, `CrossEntropyLoss` and `MSELoss`, and see why each fits a different kind of task. You will create the two optimizers from the chapter, SGD and Adam, and then combine a loss function, an optimizer, and the `DigitClassifier` into the five-stage training loop. You will train the model on the MNIST digits for several epochs, watch the loss fall, and finally measure how accurately it classifies digits it was trained on and digits it has never seen.

This is the exact workflow you would run any time you train a model in PyTorch. Every model you train later, including the convolutional model in the next chapter, follows the same loop you build here. By the end, a falling loss and a test-accuracy number will mean something specific to you, because you will have assembled every step that produces them.

## Learning Objectives

By the end of this lab, you should be able to:

- Use `requires_grad`, `.backward()`, and `.grad` to compute and read gradients for a small computation, and show why gradients must be cleared between steps  
- Compute `CrossEntropyLoss` and `MSELoss` and explain which task each one fits  
- Create an SGD optimizer and an Adam optimizer for the same model  
- Assemble and run a complete training loop (forward pass, loss computation, backward pass, parameter update, and gradient reset) for the `DigitClassifier` on MNIST  
- Evaluate a trained model on the training and test splits and interpret the accuracy gap

## Prerequisites

This course’s labs are **not cumulative**, so this lab does not depend on any prior resources or the state of earlier labs if there are any.

You need the following before you start:

- **A notebook environment:** either a Google account for Google Colab, or a local installation of Jupyter Notebook. Every step in this lab runs on a CPU; no GPU is required.  
- **Python 3.12 or later.** On Colab this is already provided. For a local environment, confirm your version with `python --version`.  
- **PyTorch 2.12.1 and TorchVision 0.17 or later (0.27 preferred),** installed with `pip` in the previous lab. If you are on Colab, these may already be present.  
- **A web browser** (Chrome, Firefox, or Edge are recommended for Colab).

## Estimated Time

45–60 minutes. Most of the hands-on time is short; the training runs in Part D add a few minutes of waiting while the model trains on CPU.

## How to Perform the Exercise

You will work entirely in one notebook, running cells from top to bottom. Create a new notebook in Google Colab (**File \> New notebook**) or a new `.ipynb` in local Jupyter. Download the chapter 3 lab notebook from the LFD2002-resources GitHub repository from the **LFD2002-resources/labs** directory. In Google Colab, select File, then Upload Notebook, and select the chapter 3 lab notebook you downloaded.

Read the explanation before running each cell, run it, and compare what you see to the Sample Output or Output shown after each block. Sample Output means the exact values will differ from run to run (random tensors, dataset counts, file paths), so match the shape and structure rather than every character. Output marks a checkpoint where a specific outcome confirms the step worked.

Commands in this lab use the notebook form, where a line beginning with \! runs a shell command from inside a notebook cell (for example, \!pip install ...). If you run these in a local terminal instead of a notebook cell, drop the leading \!.

> **Note:** Run the cells in order and only once each, unless a step says otherwise. If you restart the notebook's runtime, re-run the cells from the top, because the variables live in memory and are cleared on restart.

## Setup: Environment Preparation

The steps in this section stand up what the later sections need: the imports, a small transform, the two dataset and dataloader objects, and the model from Lab 2\. They are not the learning content of the lab, and on a managed course platform some of them may already be done for you. Run them once at the top of your notebook, then move on to Step 1\.

### Setup 1\. Import the Libraries You Will Use

Add a cell with the imports for the whole lab. You need `torch` for tensors and autograd, `torch.nn` for the loss functions, `torch.optim` for the optimizers, and two pieces from Torchvision and `torch.utils.data` to load MNIST and feed it in batches.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2


### Setup 2\. Recreate the Model Definition

The training in Part D needs the `DigitClassifier` class available in the current kernel. Before adding new code, redefine the class by running the cell below.


In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(784, 128)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

print(DigitClassifier)


**Output:**

```
<class '__main__.DigitClassifier'>
```

### Setup 3\. Load MNIST and Create DataLoaders

Training reads the dataset in small batches rather than one image at a time, so you first load the MNIST train and test splits and then wrap each in a `DataLoader` that hands out batches. A `DataLoader` is a helper that groups a dataset into batches and can shuffle them; you shuffle the training data so the model does not see the digits in the same order every epoch, and you leave the test data unshuffled because order does not matter for evaluation.

The `transform` converts each image from its stored form into a float tensor with pixel values scaled to the range 0 to 1, which is the form the model expects. Create the datasets and loaders in a new cell:


In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

train_data = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

print(f"Training images: {len(train_data)}")
print(f"Test images: {len(test_data)}")


**Output:**

```
Training images: 60000
Test images: 10000
```

The counts confirm the standard MNIST split: 60,000 training images and 10,000 test images. If you’re running locally and the `data/` directory from Lab 2 is present, this step reuses it and does not download anything; if it is missing, Torchvision downloads MNIST once.

> **Note:** The Chapter 2 material used the newer Torchvision transform pipeline. `v2.ToImage()` turns the image into a tensor, and `v2.ToDtype(torch.float32, scale=True)` casts it to float and scales the pixel values into the 0-to-1 range. This two-step pipeline is the current replacement for the older single `ToTensor()` call.

## Part A: See Autograd Compute Gradients

This part reproduces the small autograd example from the chapter so you can watch gradients appear and see why they must be cleared. No model is involved yet, just a few tensor operations in isolation.

### Step 1\. Track a Tensor and Run a Backward Pass

To make autograd compute a gradient, a tensor must be tracked, which you request by creating it with `requires_grad=True`. You then build a small computation from it, call `.backward()` on the result, and read the gradient that autograd stored in the tensor's `.grad`. Add this cell:


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x   # a computation that depends on x

y.backward()         # autograd computes the gradient of y with respect to x
print(x.grad)        # read how much y changes as x changes, near x = 2


**Output:**

```
tensor(7.)
```

The value `7.` is the gradient of `y` with respect to `x` at `x = 2`, computed by autograd from the operations you wrote. Read plainly, near `x = 2` the value of `y` rises about seven times as fast as `x`. In a real model this same mechanism produces a gradient for every parameter, which is exactly what an optimizer needs to decide which way to move each one.

### Step 2\. Watch Gradients Accumulate, Then Clear Them

Autograd adds each new gradient on top of whatever is already in `.grad` instead of replacing it. Seeing this once explains why every training step ends by clearing gradients. Run a second backward pass on a fresh computation from the same `x`, without clearing first, and watch the value grow:


In [ ]:
y2 = x ** 2 + 3 * x
y2.backward()        # no clearing first: this ADDS to the existing gradient
print(x.grad)        # 7 from before, plus 7 again


**Output:**

```
tensor(14.)
```

The gradient is now `14.`, not `7.`, because the second backward pass added another `7.` on top of the first. To get a correct, standalone gradient you clear `.grad` back to zero before computing again:


In [ ]:
x.grad.zero_()       # reset the accumulated gradient to zero
y3 = x ** 2 + 3 * x
y3.backward()
print(x.grad)


**Output:**

```
tensor(7.)
```

After clearing, the gradient is `7.` again. In the training loop you will not call `zero_()` by hand like this; the optimizer provides `optimizer.zero_grad()` to clear every parameter's gradient at once. The behavior you just saw is the reason that call exists.

## Part B: Measure Error with Loss Functions

A loss function takes predictions and targets and returns one number measuring how far off the predictions are. In this part you compute the two loss functions from the chapter and see how each behaves.

### Step 3\. Compute `CrossEntropyLoss` on Correct and Wrong Predictions

`CrossEntropyLoss` is the loss for classification. It takes the raw scores a model produces for each class, called logits, together with the correct class index, and returns a small loss when the highest score is on the correct class and a large loss when it is not. You create the loss function once and then call it. Compare a prediction that assigns the highest score to the correct class against one that assigns it to the wrong one:


In [ ]:
loss_fn = nn.CrossEntropyLoss()

# One example, three possible classes; the correct class is 0.
confident_correct = torch.tensor([[2.0, 0.5, 0.1]])   # highest score on class 0
confident_wrong   = torch.tensor([[0.1, 0.5, 2.0]])   # highest score on class 2
target = torch.tensor([0])                            # the right answer is class 0

print(loss_fn(confident_correct, target))
print(loss_fn(confident_wrong, target))


**Output:**

```
tensor(0.3168)
tensor(2.2168)
```

When the highest score was in the correct class, the loss was small. When the highest score was on a wrong class with the same score, the loss was much larger. That difference is the signal that training uses to know it did not go well. Note that you passed the raw logits directly and gave the target as a class index (`0`), not as a list of ten numbers; `CrossEntropyLoss` handles the normalization internally.

### Step 4\. Compute `MSELoss` on Continuous Values

`MSELoss` is the loss for regression, where the model predicts a continuous number rather than choosing a class. It compares predictions against same-shaped targets, squares each difference, and averages them. See the example where it is computed on three predictions and their targets:


In [ ]:
mse = nn.MSELoss()

predictions = torch.tensor([2.5, 0.0, 2.1])
targets     = torch.tensor([3.0, -0.5, 2.0])

print(mse(predictions, targets))


**Output:**

```
tensor(0.1700)
```

Unlike `CrossEntropyLoss`, `MSELoss` compares numbers of the same shape directly, with no notion of a class index. The three predictions are matched against three continuous targets, and the loss grows as they drift apart. This contrast is the practical reason to pick the loss by task type: `CrossEntropyLoss` for choosing among classes, `MSELoss` for predicting a continuous value.

### Step 5\. Compute `CrossEntropyLoss` on a Batch of Digit Predictions

Real training passes a batch of images at once, and `CrossEntropyLoss` handles a whole batch in one call. To see this, create a fresh `DigitClassifier` model,  take one batch of images and labels from the training loader, run the images through the model to get logits, and compute the loss against the labels. Because this model is untrained, expect a fairly high loss.


In [ ]:
model = DigitClassifier()

images, labels = next(iter(train_loader))   # one batch: 64 images and 64 labels
logits = model(images)                       # forward pass -> shape [64, 10]
loss = loss_fn(logits, labels)		     # computes loss - approx. 2.3 for 10 classes and an untrained model

print(f"logits shape: {logits.shape}")
print(f"labels shape: {labels.shape}")
print(f"batch loss: {loss}")


**Sample Output:**

```
logits shape: torch.Size([64, 10])
labels shape: torch.Size([64])
batch loss: 2.305
```

The logits have one row of ten scores per image, and the labels are one class index per image, which is exactly the shape pairing `CrossEntropyLoss` expects. The loss is a single number averaged across the batch. It is close to the value you would expect from an untrained model that is essentially guessing among ten classes, and lowering it is what the training loop will do. The `grad_fn` shown in the output confirms the loss is attached to the computational graph, so `.backward()` can trace from it back to the model's parameters.

## Part C: Update Parameters with an Optimizer

An optimizer reads the gradients autograd computes and uses them to adjust the model's parameters. In this part you create the two optimizers from the chapter. You will put them to work in Part D.

### Step 6\. Create an SGD Optimizer and an Adam Optimizer

You create an optimizer by handing it the parameters to train, which is what `model.parameters()` from Chapter 2 returns, along with a learning rate that sets how large each update step is. Create one of each type so you can compare them when you train:


In [ ]:
sgd_optimizer  = torch.optim.SGD(model.parameters(), lr=0.01)
adam_optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(sgd_optimizer)
print(adam_optimizer)


**Sample Output:**

```
SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

```

Both optimizers were given the same model's parameters but different learning rates, which reflects how they are typically used: plain SGD often needs a larger, well-chosen learning rate, while Adam adapts its step sizes internally and works well from a small default. You will not use both on the same model at once; in Part D you will train a fresh model with each and compare the results.

> **Note:** Printing an optimizer shows its parameter groups and settings. The exact fields shown depend on the PyTorch version, so match the shape of this output rather than every line.

## Part D: Assemble and Run the Training Loop

Here you combine everything: the forward pass, a loss function, `.backward()`, and an optimizer, run in a fixed order over many batches. You will first run one training step by hand to see each stage, then wrap the stages into a loop and train the model.

### Step 7\. Run a Single Training Step, Stage by Stage

A single training step moves one batch through four stages and one reset, in this order. Running it once by hand makes each stage visible before you hide it inside a loop. Start from a fresh model and optimizer so the step trains a clean model, take one batch, and run the five lines:


In [ ]:
model = DigitClassifier()                                  # fresh, untrained model
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

images, labels = next(iter(train_loader))

predictions = model(images)                # 1. forward pass: images -> logits
loss = loss_fn(predictions, labels)        # 2. loss: how wrong this batch was
print(f"loss before step: {loss.item():.4f}")

loss.backward()                            # 3. backward pass: fill every .grad
optimizer.step()                           # 4. update: adjust every parameter
optimizer.zero_grad()                      # 5. reset gradients for the next step

# Check the loss on the same batch after one update:
print(f"loss after step:  {loss_fn(model(images), labels).item():.4f}")


**Sample Output:**

```
loss before step: 2.3018
loss after step:  2.2740
```

The loss on the same batch dropped after a single update, which is the whole training process in miniature: the optimizer updated the parameters a small step in the direction that lowers the loss. The `.item()` call reads the single loss value out of its tensor as a plain Python number for printing. Notice the order of the last three lines matches the chapter: compute gradients, apply them, then clear them for next time.

> **Warning:** If you leave out `optimizer.zero_grad()`, the gradients from each step add onto the next step instead of being replaced, and training becomes erratic. You saw this accumulation directly in Step 5\. Keep the reset in every step.

### Step 8\. Wrap the Step into a Loop and Train for Several Epochs

To train on the whole dataset, you run those five stages for every batch in the training loader, and you repeat that full pass several times. One full pass over the training data is an epoch. Define a small training function that loops over the batches for a given number of epochs, printing the average loss at the end of each epoch so you can watch it fall:


In [ ]:
def train(model, loader, loss_fn, optimizer, epochs):
    model.train()                          # put the model in training mode
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in loader:
            predictions = model(images)          # 1. forward
            loss = loss_fn(predictions, labels)  # 2. loss
            loss.backward()                      # 3. backward
            optimizer.step()                     # 4. update
            optimizer.zero_grad()                # 5. reset
            running_loss += loss.item()
        average_loss = running_loss / len(loader)
        print(f"Epoch {epoch + 1}: average loss {average_loss:.4f}")


Now train a fresh `DigitClassifier` with the Adam optimizer for five epochs:


In [ ]:
adam_model = DigitClassifier()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(adam_model.parameters(), lr=0.001)

train(adam_model, train_loader, loss_fn, optimizer, epochs=5)


**Sample Output:**

```
Epoch 1: average loss 0.3902
Epoch 2: average loss 0.1908
Epoch 3: average loss 0.1389
Epoch 4: average loss 0.1108
Epoch 5: average loss 0.0921
```

The average loss falls each epoch, which is the visible sign that the optimizer is steadily improving the model. `model.train()` sets the model to training mode; it matters for models that contain dropout or normalization layers, and it is a good habit to include even when, as here, the model has neither. Training on CPU takes a short while per epoch, which is expected.

### Step 9\. Compare SGD with Adam

To feel the difference between the optimizers, train a second fresh model with SGD for the same five epochs and compare its loss trajectory to the Adam run. Using a fresh model matters, because training the already-trained model would not be a fair comparison.


In [ ]:
sgd_model = DigitClassifier()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(sgd_model.parameters(), lr=0.01)

train(sgd_model, train_loader, loss_fn, optimizer, epochs=5)


**Sample Output:**

```
Epoch 1: average loss 1.3921
Epoch 2: average loss 0.6215
Epoch 3: average loss 0.4790
Epoch 4: average loss 0.4136
Epoch 5: average loss 0.3735
```

With plain SGD at this modest learning rate, the loss still falls, but after five epochs it sits higher than the Adam run reached, and it started much higher in the first epoch. This matches the trade-off from the chapter: Adam adapts its step sizes per parameter and tends to make fast progress with its default learning rate, while plain SGD depends on a well-chosen learning rate and can be slower without one. Neither result is wrong; they show two optimizers making different amounts of progress in the same budget.

> **Note:** You could narrow the gap by giving SGD a larger learning rate or adding `momentum=0.9`, for example `torch.optim.SGD(sgd_model.parameters(), lr=0.1, momentum=0.9)`. Trying that is optional, but it is a direct way to see how much SGD depends on its learning-rate choice.  
> **Sample Output**

```
Epoch 1: average loss 0.2492
Epoch 2: average loss 0.1060
Epoch 3: average loss 0.0796
Epoch 4: average loss 0.0632
Epoch 5: average loss 0.0498
```

## Part E: Evaluate the Trained Model

A falling training loss shows the model is fitting the data it trained on. Evaluation checks whether it learned patterns that hold on data it has never seen. You will measure accuracy, the fraction of images classified correctly, on the test split and then on the training split.

### Step 10\. Measure Test Accuracy with Evaluation Mode and `no_grad`

Evaluation only reads predictions; it does not train. Two small changes reflect that. You switch the model into evaluation mode with `model.eval()`, and you wrap the loop in a `torch.no_grad()` block so autograd does not build the computational graph, which saves memory and time when no backward pass will follow. For each batch you take the model's highest-scoring class with `argmax` and count how many match the labels. Define an evaluation function and run it on the Adam-trained model:


In [ ]:
def evaluate(model, loader):
    model.eval()                                  # evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():                         # no gradient tracking needed
        for images, labels in loader:
            predictions = model(images)
            predicted_class = predictions.argmax(dim=1)   # highest-scoring class per image
            correct += (predicted_class == labels).sum().item()
            total += labels.size(0)
    return correct / total

test_accuracy = evaluate(adam_model, test_loader)
print(f"Test accuracy: {test_accuracy:.4f}")


**Sample Output:**

```
Test accuracy: 0.9703
```

The number is the fraction of the 10,000 test images the trained model classified correctly. `predictions.argmax(dim=1)` picks, for each image, the class with the highest logit, which is the model's answer. Comparing those answers to the true labels and dividing by the total gives accuracy.

### Step 11\. Compare Training Accuracy with Test Accuracy

Measuring accuracy on the training data as well tells you whether the model generalized or merely memorized. Run the same evaluation on the training loader and compare:


In [ ]:
train_accuracy = evaluate(adam_model, train_loader)
print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy:  {test_accuracy:.4f}")


**Sample Output:**

```
Train accuracy: 0.9787
Test accuracy:  0.9703
```

The two numbers are close, which suggests the model learned patterns that generalize rather than memorizing the training images. A large gap in the other direction, high training accuracy with much lower test accuracy, would be the overfitting you were warned about in the prerequisites. Comparing these two numbers is the standard, quick check for it.

## Cleanup

This lab does not create anything that must be torn down, and part of what it produces may be kept. There are no processes to stop and no services to remove.

**The `data/` directory** holds the downloaded MNIST files. On a local machine, keeping it means the next chapter's load is instant instead of downloading again. If you need to reclaim disk space, you may delete `data/MNIST/`, but the dataset will download again the next time you load it.

On Google Colab the runtime is temporary. When your Colab session ends, the `data/` directory and the loaded objects are cleared. This is expected. In the next chapter you simply re-run the setup cells, which downloads the data again for the new session.

## Summary

You assembled the full training chain and ran it. You watched autograd produce a gradient from `.backward()` and read it from `.grad`, and you saw gradients accumulate until cleared, which is why every step calls `optimizer.zero_grad()`. You computed `CrossEntropyLoss` and `MSELoss` and saw why the choice follows the task. You built the five-stage loop, forward, loss, backward, update, reset, trained the `DigitClassifier` on MNIST with both Adam and SGD, and compared how far each optimizer got in the same number of epochs. Finally you evaluated the trained model with `model.eval()` and `torch.no_grad()`, and compared training and test accuracy to check for overfitting.